In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.ensemble import RandomForestClassifier, IsolationForest

#### Load prepared data

In [2]:
test_results = pd.read_csv("../data/processed/test_results_with_anomaly.csv", index_col=0)
df_processed = pd.read_csv("../data/processed/creditcard_processed.csv", index_col=0)

#### Creating a new dataset by joining test results with new features from processed dataset

In [3]:
explanation_df = test_results.join(
    df_processed[["Amount", "Hour", "Log_Amount"]], how="left"
)

explanation_df.head()
explanation_df.isna().sum()

actual_class         0
fraud_probability    0
fraud_prediction     0
anomaly_score        0
anomaly_flag         0
Amount               0
Hour                 0
Log_Amount           0
dtype: int64

#### Building explanation logic

In [4]:
def generate_explanation_signals(row):
    signals = []

    if row["fraud_probability"] >= 0.9:
        signals.append("very high fraud probability")
    elif row["fraud_probability"] >= 0.7:
        signals.append("high fraud probability")
    elif row["fraud_probability"] >= 0.5:
        signals.append("moderate fraud probability")
    elif row["fraud_probability"] >= 0.1:
        signals.append("slightly elevated fraud probability")

    if row["fraud_probability"] >= 0.7 and row["anomaly_flag"] == 1:
        signals.append("multiple strong risk signals detected")
    
    if row["anomaly_flag"] == 1:
        signals.append("transaction flagged as anomalous")

    if row["anomaly_score"] < 0:
        signals.append("low anomaly score indicates unusual behavior")

    if row["Amount"] > 1000:
        signals.append("high transaction amount")
    
    if row["Hour"] >= 0 and row["Hour"] <= 5:
        signals.append("transaction occurred during late-night hours")

    return signals

explanation_df["explanation_signals"] = explanation_df.apply(generate_explanation_signals, axis=1)

#explanation_df[(explanation_df["fraud_probability"] > 0.5) & (explanation_df["anomaly_flag"] == 1 )].head(10)

#### Now building severity logic

In [5]:
def assign_severity(row):
    fraud_prob = row["fraud_probability"]
    anomaly_flag = row["anomaly_flag"]
    amount = row["Amount"]
    hour = row["Hour"]

    suspicious_count = 0

    if fraud_prob >= 0.9:
        suspicious_count += 2
    elif fraud_prob >= 0.7:
        suspicious_count += 1

    if anomaly_flag == 1:
        suspicious_count += 1
    
    if amount > 1000:
        suspicious_count += 1
    
    if 0 <= hour <= 5:
        suspicious_count += 1
    
    if fraud_prob >= 0.9 and anomaly_flag == 1:
        return "Critical"
    elif suspicious_count >= 3:
        return "High"
    elif suspicious_count >= 2:
        return "Medium"
    else:
        return "Low"
    
explanation_df["severity"] = explanation_df.apply(assign_severity, axis=1)

#explanation_df[explanation_df["severity"].isin(["Critical", "High", "Medium"])].head(10)

#### Recommended action logic

In [6]:
def recommended_action(row):
    severity = row["severity"]

    if severity == "Critical":
        return "Escalate immediately for fraud analyst review"
    elif severity == "High":
        return "Send for priority analyst review"
    elif severity == "Medium":
        return "Monitor transaction and verify if needed"
    else:
        return "No immediate action required"
    
explanation_df["recommended_action"] = explanation_df.apply(recommended_action, axis=1)

explanation_df[explanation_df["severity"].isin(["Critical", "High", "Medium"])].head(10)

,actual_class,fraud_probability,fraud_prediction,anomaly_score,anomaly_flag,Amount,Hour,Log_Amount,explanation_signals,severity,recommended_action
259482,0,8.644803e-01,1,0.141998,0,1151.88,20.0,7.050018,"[high fraud probability, high transaction amount]",Medium,Monitor transaction and verify if needed
151011,1,1.000000e+00,1,-0.085968,1,1.00,2.0,0.693147,"[very high fraud probability, multiple strong ...",Critical,Escalate immediately for fraud analyst review
136063,0,9.343708e-01,1,0.167887,0,258.99,22.0,5.560643,[very high fraud probability],Medium,Monitor transaction and verify if needed
154429,0,7.095712e-01,0,0.004728,0,3026.52,4.0,8.015499,"[high fraud probability, high transaction amou...",High,Send for priority analyst review
144686,0,3.351785e-20,0,-0.029466,1,5268.04,23.0,8.569603,"[transaction flagged as anomalous, low anomaly...",Medium,Monitor transaction and verify if needed
41501,0,9.476788e-01,1,0.167879,0,1.00,11.0,0.693147,[very high fraud probability],Medium,Monitor transaction and verify if needed
12031,0,8.072745e-01,1,0.197638,0,7.58,5.0,2.149434,"[high fraud probability, transaction occurred ...",Medium,Monitor transaction and verify if needed
155296,0,9.046571e-01,1,-0.056203,1,102.24,5.0,4.637056,"[very high fraud probability, multiple strong ...",Critical,Escalate immediately for fraud analyst review
181104,0,9.697333e-01,1,0.152854,0,417.22,10.0,6.036008,[very high fraud probability],Medium,Monitor transaction and verify if needed
114902,0,9.999927e-01,1,0.163003,0,8.99,20.0,2.301585,[very high fraud probability],Medium,Monitor transaction and verify if needed


#### Adding human readable text

In [7]:
def build_explanation_text(signals):
    if not signals:
        return "No significant suspicious signals detected."
    return "; ".join(signals).capitalize() + "."

explanation_df["explanation_text"] = explanation_df["explanation_signals"].apply(build_explanation_text)

explanation_df[explanation_df["severity"].isin(["Critical", "High", "Medium"])].head(10)

,actual_class,fraud_probability,fraud_prediction,anomaly_score,anomaly_flag,Amount,Hour,Log_Amount,explanation_signals,severity,recommended_action,explanation_text
259482,0,8.644803e-01,1,0.141998,0,1151.88,20.0,7.050018,"[high fraud probability, high transaction amount]",Medium,Monitor transaction and verify if needed,High fraud probability; high transaction amount.
151011,1,1.000000e+00,1,-0.085968,1,1.00,2.0,0.693147,"[very high fraud probability, multiple strong ...",Critical,Escalate immediately for fraud analyst review,Very high fraud probability; multiple strong r...
136063,0,9.343708e-01,1,0.167887,0,258.99,22.0,5.560643,[very high fraud probability],Medium,Monitor transaction and verify if needed,Very high fraud probability.
154429,0,7.095712e-01,0,0.004728,0,3026.52,4.0,8.015499,"[high fraud probability, high transaction amou...",High,Send for priority analyst review,High fraud probability; high transaction amoun...
144686,0,3.351785e-20,0,-0.029466,1,5268.04,23.0,8.569603,"[transaction flagged as anomalous, low anomaly...",Medium,Monitor transaction and verify if needed,Transaction flagged as anomalous; low anomaly ...
41501,0,9.476788e-01,1,0.167879,0,1.00,11.0,0.693147,[very high fraud probability],Medium,Monitor transaction and verify if needed,Very high fraud probability.
12031,0,8.072745e-01,1,0.197638,0,7.58,5.0,2.149434,"[high fraud probability, transaction occurred ...",Medium,Monitor transaction and verify if needed,High fraud probability; transaction occurred d...
155296,0,9.046571e-01,1,-0.056203,1,102.24,5.0,4.637056,"[very high fraud probability, multiple strong ...",Critical,Escalate immediately for fraud analyst review,Very high fraud probability; multiple strong r...
181104,0,9.697333e-01,1,0.152854,0,417.22,10.0,6.036008,[very high fraud probability],Medium,Monitor transaction and verify if needed,Very high fraud probability.
114902,0,9.999927e-01,1,0.163003,0,8.99,20.0,2.301585,[very high fraud probability],Medium,Monitor transaction and verify if needed,Very high fraud probability.


In [8]:
explanation_df.to_csv("../data/processed/test_results_with_explanations.csv", index=True)

#### Explanation Layer
1. A rule-based explanation engine was created to make fraud predictions more understandable.
2. The explanation logic combines fraud probability, anomaly signals, transaction amount, and transaction timing.
3. This helps translate model outputs into human-readable reasoning.

#### Severity Logic
1. Transactions are categorized into Low, Medium, High, and Critical severity levels.
2. Severity is based on the strength and number of suspicious signals.

#### Recommended Action
1. Each flagged transaction is assigned a suggested action.
2. This simulates how an AI assistant could support fraud analysts in prioritizing cases.